# EDA 
Take a look about our data in Calories Burn

In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
from  data_profiling import ProfileReport
from sklearn.model_selection import train_test_split
import seaborn as sns

## Loading data 
using `read_csv()` from `Pandas` library

In [16]:
data_path = r"\Challenges\mlops-project\src\data\raw\Student_Performance.csv"
data = pd.read_csv(data_path)

Show Sample from data use `pandas.DataFrame.sample()` not Use `pandas.DataFrame.head()` <br>
`head()` only show first but `sample()` shuffle data then take samples to show you all various in data

In [17]:
data.duplicated().sum()

np.int64(127)

In [18]:
data.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [19]:
data.columns

Index(['Hours Studied', 'Previous Scores', 'Extracurricular Activities',
       'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index'],
      dtype='object')

In [36]:
pro = ProfileReport(data, title="Data Profiling Report")

pro.to_file("data_profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:00<00:00, 2850.25it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Note <br>
Weight column --> maybe try a make two distribution from Weight Feature <br>
Body_Temp maybe this Feature have a bais <br>

## Split data
we will split data to 3 times  <br>
Train ---> dev ---> test <br>
Step 1 : train all models in Train-Set <br>
then <br> 
Step 2 : Validation all models in dev-Set <br> 
if model passed ---> compares model with others <br>
else remove it 

In [3]:
X = data.drop(columns= ["productivity_score" , 'student_id' , 'gender' , 'age'] )
y = data["productivity_score"]

In [4]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2)

In [5]:
print(f"length of X_train ---> {len(X_train)}")
print(f"length of X_test ---> {len(X_test)}")

length of X_train ---> 16000
length of X_test ---> 4000


## Data Pipeline

In [6]:
import mlflow
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder  , PowerTransformer
from sklearn.dummy import DummyRegressor
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression , Ridge , Lasso 
from sklearn.ensemble import AdaBoostRegressor , ExtraTreesRegressor

In [ ]:
ExtraTreesRegressor

In [17]:
# num_pipeline = make_pipeline( StandardScaler() )
nothing_pipeline = make_pipeline( OneHotEncoder() )
skewed_pipeline = make_pipeline( StandardScaler() , PowerTransformer() )

skewed_columns = ["Social_Media_Hours_Per_Day" , "Study_Hours_Per_Week" , "Sleep_Hours_Per_Night"]
nothing_columns = ["Stress_Level" , "AI_Tool_Usage_Frequency"]
num_columns = ['']

processor = ColumnTransformer(transformers=[
    # ( "num pipeline", num_pipeline , num_columns) , 
    ("nothing" , nothing_pipeline , nothing_columns),
    ("skewed" , skewed_pipeline , skewed_columns)
],remainder="passthrough")

Baseline = Pipeline( [ ("Preprocessing" , processor )] )
Baseline



X_train_transed = processor.fit_transform(X_train , y_train )
X_test_transed = processor.fit_transform(X_test , y_test)

In [18]:
Baseline.fit(X_train , y_train)
Baseline.score(X_train , y_train)

AttributeError: This 'Pipeline' has no attribute 'score'

In [102]:
Baseline.score(X_test , y_test)

0.5347420908698095

In [7]:
from scipy.stats import loguniform

param_distributions = {
    "alpha": loguniform(1e-4, 100),
    "selection": ["cyclic", "random"],
    "tol": loguniform(1e-5, 1e-2),
    "fit_intercept": [True, False],
    "max_iter": [2000, 5000, 10000, 20000],
}

In [8]:
etree = Lasso(
    random_state=42,
)

random_search = RandomizedSearchCV(
    estimator=etree,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

In [9]:
etree.fit(X_train , y_train)
etree.score(X_train , y_train)

0.9976836715860162

In [10]:
etree.score(X_test , y_test)

0.9976941217582747

In [11]:

from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor,
)
from sklearn.model_selection import RandomizedSearchCV

models_params = {
    "ExtraTrees": {
        "model": ExtraTreesRegressor(random_state=42, n_jobs=1),
        "params": {
            "n_estimators": randint(100, 601),
            "max_depth": [None, 5, 10, 15, 20, 30],
            "min_samples_split": randint(2, 16),
            "min_samples_leaf": randint(1, 9),
            "max_features": [1.0, "sqrt", "log2", 0.5, 0.8],
        },
    },

    "RandomForest": {
        "model": RandomForestRegressor(random_state=42, n_jobs=1),
        "params": {
            "n_estimators": randint(100, 601),
            "max_depth": [None, 5, 10, 15, 20, 30],
            "min_samples_split": randint(2, 16),
            "min_samples_leaf": randint(1, 9),
            "max_features": [1.0, "sqrt", "log2", 0.5, 0.8],
        },
    },

    "HistGradientBoosting": {
        "model": HistGradientBoostingRegressor(random_state=42),
        "params": {
            "learning_rate": loguniform(0.01, 0.3),
            "max_iter": randint(100, 501),
            "max_leaf_nodes": randint(7, 32),
            "max_depth": [None, 3, 5, 8, 12],
            "min_samples_leaf": randint(5, 31),
            "l2_regularization": loguniform(1e-4, 10),
        },
    },

    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            "n_estimators": randint(100, 501),
            "learning_rate": loguniform(0.01, 0.2),
            "max_depth": randint(2, 6),
            "min_samples_split": randint(2, 16),
            "min_samples_leaf": randint(1, 9),
            "subsample": uniform(0.6, 0.4),
            "max_features": [None, "sqrt", "log2"],
        },
    },
}

results = {}

for name, config in models_params.items():
    search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=30,
        scoring="r2",
        cv=5,
        n_jobs=-1,
        random_state=42,
        return_train_score=True,
        verbose=1,
    )

    search.fit(X_train, y_train)

    results[name] = search

    print(f"\n{name}")
    print("Best CV R2:", search.best_score_)
    print("Best Params:", search.best_params_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits

ExtraTrees
Best CV R2: 0.9808204009190318
Best Params: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 7, 'n_estimators': 352}
Fitting 5 folds for each of 30 candidates, totalling 150 fits

RandomForest
Best CV R2: 0.973068477523778
Best Params: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 7, 'n_estimators': 352}
Fitting 5 folds for each of 30 candidates, totalling 150 fits

HistGradientBoosting
Best CV R2: 0.9970428044126571
Best Params: {'l2_regularization': np.float64(0.8930539642879287), 'learning_rate': np.float64(0.07853737079666878), 'max_depth': 3, 'max_iter': 330, 'max_leaf_nodes': 15, 'min_samples_leaf': 11}
Fitting 5 folds for each of 30 candidates, totalling 150 fits

GradientBoosting
Best CV R2: 0.9974728458484738
Best Params: {'learning_rate': np.float64(0.06490876477881215), 'max_depth': 3, 'max_features': None, 'min_samples_leaf': 5

In [12]:
LR = LinearRegression()
LR.fit(X_train , y_train)
LR.score(X_train , y_train)

0.9999999682365832

In [13]:
LR.score(X_test , y_test)

0.9999999680834775

In [4]:
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from pandas import DataFrame
from typing import List




def Preprocess(Category:List , X:DataFrame , y:DataFrame = None ) -> DataFrame :
    """ ## Inputs
    `X` ,`y` Prepare them For Training 
    ## outputs 
    X_train , y """
    num_pipeline = make_pipeline( StandardScaler() )
    category_pipeline = make_pipeline( OneHotEncoder(drop="if_binary", sparse_output=False,handle_unknown="ignore") )

    category_columns = Category
    num_columns = X.columns.difference(category_columns).tolist()

    pipes = ColumnTransformer(transformers=[
        ("num pipe" , num_pipeline , num_columns) ,
        ("category pipe" , category_pipeline , category_columns)
    ])
    processor = Pipeline([
        ( "preprocessor ", pipes) , 
    ]).set_output(transform="pandas")

    return processor.fit_transform(X)


In [12]:
import joblib
import pandas as pd
model = joblib.load(r"../models/ada.pkl")

In [37]:
def predict(hours_studied , previous_scores , extracurricular ,sleep_hours , sample_papers ):
    input_data = pd.DataFrame(
        [{
            "Hours Studied": hours_studied,
            "Previous Scores": previous_scores,
            "Extracurricular Activities": extracurricular,
            "Sleep Hours": sleep_hours,
            "Sample Question Papers Practiced": sample_papers,
        }] )

    transed = Preprocess(Category=['Extracurricular Activities'] , X =input_data)
    return model.predict(transed)

In [41]:
predict(hours_studied=8 , previous_scores = 100, extracurricular= "No",sleep_hours= 4, sample_papers=7 )

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- category pipe__Extracurricular Activities_No
Feature names seen at fit time, yet now missing:
- category pipe__Extracurricular Activities_Yes


# Load Model

In [9]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")

model = mlflow.sklearn.load_model(
    "models:/GPA-Predict/11"
)

In [11]:
model_path = mlflow.artifacts.download_artifacts(
    artifact_uri=f"runs:/decbbddf8a9545bfad09dbf43a5b99f5/BaggingRegressor"
)

In [12]:
model_path

'C:\\Users\\Ahmed\\AppData\\Local\\Temp\\tmpu4yhvn30\\BaggingRegressor'

In [13]:
import mlflow.sklearn

model = mlflow.sklearn.load_model(model_path)

In [14]:
model

,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.7
,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [ ]:
import joblib
pip= joblib.load(r"D:\Challenges\mlops-project\src\models\dumps\preprocessor.pkl")

In [1]:
import glob
path = r'../data/raw/'
files = glob.glob(path + '*.csv')


In [2]:
len(files)

1

In [ ]:
import pandas as pd

df = pd.read_csv(files[0] , )

In [6]:
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no
